# Reporte EDA del Sprint 1 (HU02)
**Proyecto:** Modelo predictivo basado en XGBoost para la identificación temprana del riesgo de incumplimiento en créditos a microempresas unipersonales  
**Épica:** EP01 Preparación y calidad de datos | **Tareas:** TK-02.1 a TK-02.5  
**Fuente:** último dataset base versionado por HU01 (`data/raw/`)

In [ ]:
import glob, sys, json
sys.path.insert(0, '..')
import pandas as pd
from IPython.display import Image, display, Markdown
from training.common import load_contract
from training.eda import run_eda
import logging; logging.getLogger('eda').setLevel(logging.WARNING)
pd.set_option('display.float_format', '{:,.2f}'.format)
contract = load_contract()
src = sorted(glob.glob('../data/raw/dataset_base_*.parquet'))[-1]
df = pd.read_parquet(src)
res = run_eda(df, contract, out_dir='../reports/eda')
s = res['summary']
print(f'Dataset: {src.split("/")[-1]} | {len(df):,} filas x {df.shape[1]} columnas')

## 1. Resumen ejecutivo

In [ ]:
o = s['objetivo']
display(Markdown(f"""| Indicador | Valor |
|---|---|
| Solicitudes analizadas | {o['n_total']:,} |
| Tasa de incumplimiento | {o['tasa_incumplimiento']:.2%} |
| Ratio de desbalance (0:1) | {o['ratio_desbalance']}:1 |
| Variables con faltantes | {s['variables_con_faltantes']} (máx. {s['max_pct_faltantes']:.1f} %) |
| Duplicados exactos | {s['duplicados']:,} |
| Valores fuera de rango de negocio | {s['inconsistencias'] - s['duplicados']:,} |"""))

## 2. Estadísticas descriptivas (TK-02.1)

In [ ]:
res['stats']

## 3. Variable objetivo (TK-02.2)

In [ ]:
display(Image(s['figuras']['objetivo'], width=420))

### 3.1 Análisis temporal (feedback del PO)

In [ ]:
display(Image(s['figuras']['temporal'], width=760))

## 4. Valores faltantes, atípicos e inconsistencias (TK-02.3)

In [ ]:
display(Image(s['figuras']['faltantes'], width=520))
res['outliers'].head(8)

In [ ]:
res['consistency'][res['consistency'].n_inconsistentes > 0]

## 5. Correlaciones y relación con el incumplimiento (TK-02.4)

In [ ]:
display(Image(s['figuras']['correlaciones'], width=640))
display(Image(s['figuras']['boxplots'], width=760))

## 6. Hallazgos y recomendaciones para HU03

In [ ]:
top = ', '.join(f'{k} ({v:+.3f})' for k, v in s['top5_corr_objetivo'].items())
m = res['missing']; con_f = m[m.pct_faltantes > 0]
alta_asim = res['stats'][res['stats'].asimetria.abs() > 1].index.tolist()
display(Markdown(f'''
1. **Desbalance:** {o['tasa_incumplimiento']:.2%} de incumplidos (ratio {o['ratio_desbalance']}:1). Justifica SMOTE-ENN solo en entrenamiento (HU06) y métricas robustas (AUC, Recall, F1, KS).
2. **Faltantes:** {len(con_f)} variables con faltantes; la mayor es `{con_f.index[0]}` ({con_f.pct_faltantes.iloc[0]:.1f} %). Se imputará con mediana de entrenamiento más indicador de faltante.
3. **Asimetría:** {len(alta_asim)} variables con |asimetría| > 1 ({', '.join(alta_asim[:5])}); se prefiere mediana sobre media.
4. **Inconsistencias:** {s['inconsistencias'] - s['duplicados']:,} valores fuera de rango y {s['duplicados']:,} duplicados: se convierten a nulo y se eliminan, respectivamente.
5. **Señal predictiva preliminar (|Spearman| con el objetivo):** {top}. Ninguna variable aislada domina, lo que respalda un modelo no lineal (XGBoost) y la selección con Boruta/RFECV (Sprint 2).
'''))